# Nonlinearity Analysis Notebook

This notebook walks you through an example of running the nonlinearity analysis pipeline. We begin by processing the images using my separate `pedestal_subtraction` package. The pedestal subtraction consists of stitching several equal-parameter images together, subtracting the pedestal along one or two axes, then calibrating the CCDs. We then run the nonlinearity analysis on the pedestal subtracted images: we use a scipy peak finding algorithm to find all the peaks in the pixel charge distribution then computing the nonlinearity, defined as the difference between the measured charge in electrons at each peak and the peak index. We then fit the curve of nonlinearity versus measured charge to a parabola and compute the interpolated nonlinearity at 500 electrons, which we use as a standard benchmark for comparing between datasets.

We also explore addition optional processes that the nonlinearity package is capable of, including correcting each pixel value in the images using the fitted nonlinearity curve, as well as some prototype gain-independent nonlinearity algorithms.

In [ ]:
base_config_file = "nonlinearity.json"

images = "**/deltaV_H_scan_VR-3_VDD-20/VR-3_VDD-20_delta_H_3/cts_avg*"

stitch_fits = True

In [ ]:
from nonlinearity_studies import notebook as nb
from nonlinearity_studies.plotting import plot_images

args, res, figs = nb.run_standard(base_config_file, images, stitch_fits)
titles = dict(additional_title=args.extra_plot_title, nimages=res['nimages'], show_titles=args.show_titles)

## Stitching the images
First, the `n` images are stitched vertically, as in along the rows, per extension. In this example, a directory of five images all with the same parameters and image dimensions are supplied. Each image is 250x2500 pixels with 500 skips and 1x1 binning and takes 4 hours to read out.

In [ ]:
n = res['nimages']
nb.show_panel(nb.figures_of(plot_images, [res['raw_data_ext']], suptitle=f"{n} Stitched Image{'s' * (n != 1)}" if n else 'Stitched Image',
                            individual_figsize=(15, 7.5), **titles))

## Identifying hot columns
Next, we identify hot columns, defined as columns with median charge ≥ 1.5 standard deviations from the median pedestal. These columns, whose data points are colored red and shaded in grey, are masked in the pedestal subtraction and calibration process.

In [ ]:
nb.show_panel(figs['plot_charge_per_column'])

## Pedestal subtraction and calibration
Here, we plot the zero and one electron peaks in both ADU and electrons. The charge spectrum is plotted in blue and the fitted double Gaussian curve is shown in red. The readout noise is defined as the shared standard deviation σ and the gain is defined as the differece between the peak locations of the first and zeroth electron peaks (μ_1 - μ_2). In the ADU plot, σ, μ_1 and μ_2 are given in ADU.

The electrons plot is generated by dividing each pixel charge by the gain. The double Gaussian is scaled from the ADU fit using the gain.

In [ ]:
nb.zero_one_explorer(res, 'ADU')

In [ ]:
nb.zero_one_explorer(res, 'electrons')

## Find all peaks
Next, we use the scipy peak finding algorithm to find all peaks in the pixel charge distribution in electrons. The charge spectrum is plotted in blue, peak locations are indicated by dashed red vertical lines and corresponding peak indices.

In [ ]:
nb.all_peaks_explorer(res)

## Fit nonlinearity curve
Finally, we plot the nonlinearity curve: the peak index subtracted from the measured pixel charge at each peak location versus the measured pixel charge. The data are shown in blue scatter points and the parabolic curve of best fit is shown in red, with its equation given in the legend.

In [ ]:
nb.show_panel(figs['plot_nonlinearity'])

## Optional additional processes

### Create nonlinearity corrected images 
In the next two cells, we demonstrate how we can correct the image we just analyzed using the interpolated nonlinearity at each pixel charge value.

In [ ]:
original_e, corrected_e, res_corrected = nb.correct_nonlinearity(args, res)

First, we compare side-by-side the original stitched image to the nonlinearity corrected image. The corrected version is computed by subtracting the interpolated nonlinearity from each pixel charge value.

In [ ]:
from matplotlib.colors import SymLogNorm

nb.show_panel(nb.figures_of(plot_images, [original_e, corrected_e], ['Original', 'NL-corrected'],
                            stack='vertical', individual_figsize=(15, 11), norm=SymLogNorm, norm_kwargs=dict(linthresh=1),
                            cbar_label=r'Charge ($e^-$)', suptitle='Nonlinearity Correction', **titles))

Then we plot, on the same axes, the nonlinearity curves for the original and corrected images curve and original image. The original nonlinearity data are plotted in blue and the quadratic curve of best fit is shown in red; the corrected image is be plotted in purple and fit to an orange quadratic curve.

In [ ]:
nb.show_panel(nb.figures_of(nb.plot_nonlinearity_comparison, args, res, res_corrected))

### Gain independent nonlinearity plots
Now we show some of the gain independent alternatives to the nonlinearity analysis.

In [ ]:
gain_indep_nonlin_config = 'gain_indep_nonlinearity.json'
gi_args, gi_res, gi_figs = nb.run_gain_independent(gain_indep_nonlin_config, res['fits_path'])

In [ ]:
for name, pngs in gi_figs.items():
    nb.show_panel(pngs, label=name)